# HBAAC 2026: End-to-End Pipeline

Notebook này chứa toàn bộ đường ống xử lý dữ liệu từ đầu đến cuối, bao gồm:
1. Tự động tải dữ liệu từ Google Drive qua gdown
2. Làm sạch dữ liệu chặt chẽ (Data Cleaning)
3. Tạo lưới thời gian liên tục và Kỹ nghệ Đặc trưng
4. Huấn luyện mô hình LightGBM chặn sớm
5. Suy luận và xuất kết quả định dạng chuẩn cuộc thi

## 1. Khởi Tạo Thư Viện

In [2]:
import os
import gc
import logging
from pathlib import Path
from typing import Dict, Tuple
import numpy as np
import pandas as pd
import lightgbm as lgb
import gdown
import warnings
warnings.filterwarnings('ignore')

## 2. Tự Động Tải Dữ Liệu
Khởi tạo cấu trúc thư mục làm việc cục bộ và nạp dữ liệu từ kho lưu trữ đám mây chia sẻ công khai.

In [3]:
os.makedirs('data/raw', exist_ok=True)
os.makedirs('submissions', exist_ok=True)

train_id = 'Dien_ID_File_Train_Cua_Ban_Vao_Day'
sample_id = 'Dien_ID_File_Sample_Submission_Cua_Ban_Vao_Day'

gdown.download(f'https://drive.google.com/uc?id=1NWxS9Z-U54ib0xDP1VefHtam1GRbYbA_', 'data/raw/train.csv', quiet=False)
gdown.download(f'https://drive.google.com/uc?id=1JG-Z2RowstWQCgEbI4VFR45pxmlc3U5k', 'data/raw/sample_submission.csv', quiet=False)

DEFAULT_TRAIN_PATH = "data/raw/train.csv"
DEFAULT_SAMPLE_PATH = "data/raw/sample_submission.csv"

Downloading...
From: https://drive.google.com/uc?id=1NWxS9Z-U54ib0xDP1VefHtam1GRbYbA_
To: d:\Study source\HBAAC\HBAAC---Last-Day\notebooks\data\raw\train.csv
100%|██████████| 45.8M/45.8M [00:01<00:00, 31.5MB/s]
Downloading...
From: https://drive.google.com/uc?id=1JG-Z2RowstWQCgEbI4VFR45pxmlc3U5k
To: d:\Study source\HBAAC\HBAAC---Last-Day\notebooks\data\raw\sample_submission.csv
100%|██████████| 2.49M/2.49M [00:00<00:00, 13.4MB/s]


## 3. Kiến Trúc Mô-đun Làm Sạch Dữ Liệu
Định nghĩa các hàm xử lý dữ liệu thô, loại bỏ mâu thuẫn hệ thống ERP, tối ưu hóa bộ nhớ an toàn và phòng chống lỗi sập ngầm.

In [4]:
logger = logging.getLogger(__name__)
SAMPLE_ID_COLUMN = "id"
SAMPLE_ID_SUFFIXES = ("_validation", "_evaluation")

def normalize_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = (
        df.columns.astype(str)
        .str.strip()
        .str.replace(" ", "_", regex=False)
    )
    return df

def parse_decimal_comma(series: pd.Series) -> pd.Series:
    cleaned = (
        series.astype(str)
        .str.strip()
        .str.replace(" ", "", regex=False)
        .str.replace(",", ".", regex=False)
    )
    cleaned = cleaned.replace({
        "": np.nan,
        "nan": np.nan,
        "None": np.nan,
    })
    return pd.to_numeric(cleaned, errors="coerce")

def extract_sample_skus(sample: pd.DataFrame) -> pd.Series:
    if SAMPLE_ID_COLUMN not in sample.columns:
        raise ValueError(f"Missing column: {SAMPLE_ID_COLUMN}")
    skus = sample[SAMPLE_ID_COLUMN].astype(str).str.strip()
    for suffix in SAMPLE_ID_SUFFIXES:
        skus = skus.str.replace(suffix, "", regex=False)
    result = skus.drop_duplicates().reset_index(drop=True)
    if result.empty:
        logger.warning("No SKUs extracted from sample submission. Check sample format.")
    return result

def reduce_mem_usage(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    start_mem = df.memory_usage(deep=True).sum() / 1024**2
    for col in df.columns:
        col_type = df[col].dtype
        if pd.api.types.is_datetime64_any_dtype(col_type):
            continue
        if pd.api.types.is_integer_dtype(col_type):
            c_min = df[col].min()
            c_max = df[col].max()
            if c_min >= np.iinfo(np.int8).min and c_max <= np.iinfo(np.int8).max:
                df[col] = df[col].astype(np.int8)
            elif c_min >= np.iinfo(np.int16).min and c_max <= np.iinfo(np.int16).max:
                df[col] = df[col].astype(np.int16)
            elif c_min >= np.iinfo(np.int32).min and c_max <= np.iinfo(np.int32).max:
                df[col] = df[col].astype(np.int32)
            else:
                df[col] = df[col].astype(np.int64)
        elif pd.api.types.is_float_dtype(col_type):
            if col in ["UnitPrice", "SalesAmount", "Unit_Cost", "Cost_Amount"]:
                continue
            c_min = df[col].min()
            c_max = df[col].max()
            if (
                np.isfinite(c_min)
                and np.isfinite(c_max)
                and c_min >= np.finfo(np.float32).min
                and c_max <= np.finfo(np.float32).max
            ):
                df[col] = df[col].astype(np.float32)
            else:
                df[col] = df[col].astype(np.float64)
    end_mem = df.memory_usage(deep=True).sum() / 1024**2
    logger.info(f"Memory reduced: {start_mem:.2f} MB -> {end_mem:.2f} MB")
    return df

def read_and_clean_transactions(
    train_path: str = DEFAULT_TRAIN_PATH,
    sample_path: str = DEFAULT_SAMPLE_PATH,
    drop_exact_duplicates: bool = False,
) -> Tuple[pd.DataFrame, pd.DataFrame, Dict]:
    raw = pd.read_csv(train_path, dtype=str)
    sample = pd.read_csv(sample_path, dtype=str)
    initial_rows = int(len(raw))
    raw = normalize_columns(raw)
    required_cols = [
        "Date",
        "Stt",
        "ItemCode",
        "Quantity",
        "UnitPrice",
        "SalesAmount",
        "Unit_Cost",
        "Cost_Amount",
    ]
    missing_cols = [
        c for c in required_cols
        if c not in raw.columns
    ]
    if missing_cols:
        raise ValueError(f"Missing required columns: {missing_cols}")
    raw["Date"] = pd.to_datetime(raw["Date"], errors="coerce")
    raw["Stt"] = raw["Stt"].astype(str).str.strip()
    raw["ItemCode"] = raw["ItemCode"].astype(str).str.strip()
    numeric_cols = [
        "Quantity",
        "UnitPrice",
        "SalesAmount",
        "Unit_Cost",
        "Cost_Amount",
    ]
    for col in numeric_cols:
        raw[col] = parse_decimal_comma(raw[col])
    duplicate_subset = [
        "Date",
        "ItemCode",
        "Quantity",
        "SalesAmount",
        "Cost_Amount",
    ]
    raw["is_exact_duplicate" ] = raw.duplicated(subset=duplicate_subset, keep="first").astype(np.int8)
    total_exact_duplicates = int(raw["is_exact_duplicate"].sum())
    if drop_exact_duplicates:
        raw = raw.loc[raw["is_exact_duplicate"].eq(0)].copy()
    raw["bad_date"] = raw["Date"].isna().astype(np.int8)
    raw["bad_item_code"] = (
        raw["ItemCode"].isna()
        | raw["ItemCode"].eq("")
        | raw["ItemCode"].str.lower().eq("nan")
    ).astype(np.int8)
    for col in numeric_cols:
        raw[f"bad_{col}"] = raw[col].isna().astype(np.int8)
    critical_bad = (
        raw["bad_date"].eq(1)
        | raw["bad_item_code"].eq(1)
        | raw["bad_Quantity"].eq(1)
        | raw["bad_UnitPrice"].eq(1)
        | raw["bad_SalesAmount"].eq(1)
        | raw["bad_Cost_Amount"].eq(1)
        | raw["bad_Unit_Cost"].eq(1)
    )
    clean = raw.loc[~critical_bad].copy()
    clean["is_zero_quantity"] = (clean["Quantity"] == 0).astype(np.int8)
    clean["is_return"] = (clean["Quantity"] < 0).astype(np.int8)
    clean["is_sign_mismatch" ] = (
        (
            (clean["Quantity"] < 0)
            & (
                (clean["SalesAmount"] > 0)
                | (clean["Cost_Amount"] > 0)
            )
        )
        | (
            (clean["Quantity"] > 0)
            & (
                (clean["SalesAmount"] < 0)
                | (clean["Cost_Amount"] < 0)
            )
        )
    ).astype(np.int8)
    tolerance = 0.01
    with np.errstate(divide="ignore", invalid="ignore"):
        implied = np.where(
            clean["Quantity"] != 0,
            clean["SalesAmount"] / clean["Quantity"],
            np.nan,
        )
    clean["is_price_mismatch"] = (
        pd.Series(implied, index=clean.index).sub(clean["UnitPrice"]).abs() > tolerance
    ).astype(np.int8)
    sample_skus = set(extract_sample_skus(sample))
    clean["sku_in_sample"] = clean["ItemCode"].isin(sample_skus).astype(np.int8)
    report = {
        "initial_rows": initial_rows,
        "rows_after_cleaning": int(len(clean)),
        "clean_ratio": float(len(clean) / max(initial_rows, 1)),
        "unique_skus": int(clean["ItemCode"].nunique()),
        "date_min": str(clean["Date"].min().date()) if not clean.empty else None,
        "date_max": str(clean["Date"].max().date()) if not clean.empty else None,
        "exact_duplicate_rows": total_exact_duplicates,
        "zero_quantity_rows": int(clean["is_zero_quantity"].sum()),
        "return_rows": int(clean["is_return"].sum()),
        "sign_mismatch_rows": int(clean["is_sign_mismatch"].sum()),
        "price_mismatch_rows": int(clean["is_price_mismatch"].sum()),
    }
    clean = reduce_mem_usage(clean)
    gc.collect()
    return clean, sample, report

## 4. Thực Thi Đường Ống Làm Sạch Dữ Liệu

In [5]:
clean_df, sample_sub, report = read_and_clean_transactions(
    train_path=DEFAULT_TRAIN_PATH,
    sample_path=DEFAULT_SAMPLE_PATH,
    drop_exact_duplicates=True
)
print(report)

{'initial_rows': 711980, 'rows_after_cleaning': 673690, 'clean_ratio': 0.9462203994494227, 'unique_skus': 15972, 'date_min': '2020-11-17', 'date_max': '2025-09-05', 'exact_duplicate_rows': 38290, 'zero_quantity_rows': 3044, 'return_rows': 37100, 'sign_mismatch_rows': 31, 'price_mismatch_rows': 289892}


## 5. Xây Dựng Lưới Lịch Liên Tục & Kỹ Nghệ Đặc Trưng
Gộp nhóm dữ liệu lượng bán thực tế sang lưới thời gian liên tục cấp độ ngày để bảo toàn ý nghĩa khoảng cách vật lý khi tạo các biến trễ (Lag) và biến động trượt.

In [7]:
TARGET_HORIZON = 56
MIN_LAG = 56
LAGS = [56, 63, 70, 84]
ROLL_WINDOWS = [7, 14, 28, 56]
WARMUP = 84

# Only positive Quantity (no returns)
sales_df = clean_df[(clean_df["Quantity"] > 0) & (clean_df["sku_in_sample"] == 1)].copy()
daily_sales = sales_df.groupby(["Date", "ItemCode"], as_index=False)["Quantity"].sum()
daily_sales["Date"] = pd.to_datetime(daily_sales["Date"])

sample_submission = pd.read_csv("data/raw/sample_submission.csv")
all_expected_skus = (
    sample_submission["id"]
    .str.replace("_validation", "", regex=False)
    .str.replace("_evaluation", "", regex=False)
    .unique()
)

TRAIN_END = daily_sales["Date"].max()

full_dates = pd.date_range(
    start=daily_sales["Date"].min(),
    end=TRAIN_END + pd.Timedelta(days=TARGET_HORIZON),
    freq="D"
)

df = pd.MultiIndex.from_product(
    [all_expected_skus, full_dates],
    names=["ItemCode", "Date"]
).to_frame(index=False)

df = df.merge(daily_sales, on=["Date", "ItemCode"], how="left")

# Label future rows
df["is_future"] = (df["Date"] > TRAIN_END).astype(bool)

# For training dates: fill NaN sales with 0
df.loc[~df["is_future"], "Quantity"] = df.loc[~df["is_future"], "Quantity"].fillna(0)
df["Quantity"] = df["Quantity"].astype(np.float32)

df = df.sort_values(["ItemCode", "Date"]).reset_index(drop=True)

# SKU-level statistics (training data only — no leakage)
sku_stats = (
    daily_sales[daily_sales["Date"] <= TRAIN_END]
    .groupby("ItemCode")["Quantity"]
    .agg(sku_mean="mean", sku_std="std")
    .reset_index()
)
sku_stats["sku_std"] = sku_stats["sku_std"].fillna(0)
sku_stats["sku_cv"] = sku_stats["sku_std"] / (sku_stats["sku_mean"] + 1e-6)

df = df.merge(sku_stats[["ItemCode", "sku_mean", "sku_std", "sku_cv"]], on="ItemCode", how="left")
df[["sku_mean", "sku_std", "sku_cv"]] = df[["sku_mean", "sku_std", "sku_cv"]].fillna(0)

# Compute lag/rolling features using qty_for_lag (future dates remain NaN)
df["qty_for_lag"] = df["Quantity"].copy()
g = df.groupby("ItemCode")["qty_for_lag"]

for lag in LAGS:
    df[f"lag_{lag}"] = g.shift(lag).astype(np.float32)

for w in ROLL_WINDOWS:
    df[f"roll_mean_{w}"] = g.transform(
        lambda x: x.shift(MIN_LAG).rolling(w, min_periods=1).mean()
    ).astype(np.float32)
    df[f"roll_std_{w}"] = g.transform(
        lambda x: x.shift(MIN_LAG).rolling(w, min_periods=2).std().fillna(0)
    ).astype(np.float32)

df["trend_7_28"] = (df["roll_mean_7"] / (df["roll_mean_28"] + 1e-6)).clip(0, 10)
df["volatility"] = (df["roll_std_28"] / (df["roll_mean_28"] + 1e-6)).clip(0, 10)
df["season_strength_7_56"] = (df["roll_mean_7"] / (df["roll_mean_56"] + 1e-6)).clip(0, 10)

# Zero-rate: use training Quantity only
df["is_zero"] = (df["Quantity"].fillna(1) == 0).astype(np.int8)
df.loc[df["is_future"], "is_zero"] = np.nan
df["zero_rate_28"] = (
    df.groupby("ItemCode")["is_zero"]
    .transform(lambda x: x.shift(MIN_LAG).rolling(28, min_periods=1).mean())
    .astype(np.float32)
)

# Days since last sale (lagged by MIN_LAG to avoid target leakage)
qty_train = df["Quantity"].copy()
qty_train[df["is_future"]] = np.nan
df["last_sale_date_actual"] = (
    df.groupby("ItemCode")["Date"]
    .transform(lambda dates: dates.where(qty_train.loc[dates.index] > 0).ffill())
)
df["days_since_last_sale_actual"] = (df["Date"] - df["last_sale_date_actual"]).dt.days

df["days_since_last_sale"] = (
    df.groupby("ItemCode")["days_since_last_sale_actual"]
    .shift(MIN_LAG) + MIN_LAG
).fillna(365).astype(np.int16)

# Calendar
df["dow"] = df["Date"].dt.dayofweek.astype(np.int8)
df["month"] = df["Date"].dt.month.astype(np.int8)
df["week"] = df["Date"].dt.isocalendar().week.astype(np.int16)
df["year"] = df["Date"].dt.year.astype(np.int16)
df["day_of_year"] = df["Date"].dt.dayofyear.astype(np.int16)
df["dow_sin"] = np.sin(2 * np.pi * df["dow"] / 7).astype(np.float32)
df["dow_cos"] = np.cos(2 * np.pi * df["dow"] / 7).astype(np.float32)
df["month_sin"] = np.sin(2 * np.pi * df["month"] / 12).astype(np.float32)
df["month_cos"] = np.cos(2 * np.pi * df["month"] / 12).astype(np.float32)
df["is_weekend"] = (df["dow"] >= 5).astype(np.int8)
df["is_month_start"] = (df["Date"].dt.day <= 3).astype(np.int8)
df["is_month_end"] = (df["Date"].dt.day >= 28).astype(np.int8)

df["ItemCode_cat"] = df["ItemCode"].astype("category")

# Keep numeric columns finite
num_cols = df.select_dtypes(include=[np.number]).columns
df[num_cols] = df[num_cols].replace([np.inf, -np.inf], np.nan)

features = [
    "dow", "month", "week", "year", "day_of_year",
    "dow_sin", "dow_cos", "month_sin", "month_cos",
    "is_weekend", "is_month_start", "is_month_end",
    "lag_56", "lag_63", "lag_70", "lag_84",
    "roll_mean_7", "roll_mean_14", "roll_mean_28", "roll_mean_56",
    "roll_std_7", "roll_std_14", "roll_std_28", "roll_std_56",
    "trend_7_28", "volatility", "season_strength_7_56",
    "zero_rate_28", "days_since_last_sale",
    "sku_mean", "sku_std", "sku_cv",
    "ItemCode_cat",
]

# Split train/val and prepare df_model
date_min = df["Date"].min()
df_model = df[
    (~df["is_future"]) &
    (df["Date"] >= date_min + pd.Timedelta(days=WARMUP))
].copy()
df_model["target"] = df_model["Quantity"]
df_model = df_model.dropna(subset=features + ["target"])

target = "target"


## 6. Thiết Kế Tập Huấn Luyện & Mô Hình Hóa Với LightGBM
Phân chia tập huấn luyện và tập kiểm định nghiêm ngặt theo trục thời gian (Temporal Validation Split) để chống rò rỉ dữ liệu.

In [ ]:
import lightgbm as lgb
import numpy as np
from sklearn.metrics import mean_squared_error

df_model = df_model.sort_values(['ItemCode', 'Date'])

X = df_model[features]
y = df_model[target]

VAL_DAYS = 56
train_cutoff = df_model["Date"].max() - pd.Timedelta(days=VAL_DAYS)
train_mask = df_model['Date'] <= train_cutoff
val_mask = df_model['Date'] > train_cutoff

X_train = X[train_mask]
y_train = y[train_mask]
X_val = X[val_mask]
y_val = y[val_mask]

train_data = lgb.Dataset(
    X_train,
    label=y_train,
    categorical_feature=['ItemCode_cat'],
    free_raw_data=False
)

val_data = lgb.Dataset(
    X_val,
    label=y_val,
    reference=train_data,
    categorical_feature=['ItemCode_cat'],
    free_raw_data=False
)

params = {
    "objective":              "tweedie",
    "tweedie_variance_power": 1.1,
    "metric":                 "tweedie",
    "learning_rate":          0.03,
    "num_leaves":             255,
    "max_depth":              -1,
    "feature_fraction":       0.8,
    "bagging_fraction":       0.85,
    "bagging_freq":           5,
    "lambda_l1":              0.05,
    "lambda_l2":              0.1,
    "min_data_in_leaf":       50,
    "seed":                   42,
    "verbose":               -1,
    "n_jobs":                -1,
}

model = lgb.train(
    params,
    train_data,
    num_boost_round=5000,
    valid_sets=[train_data, val_data],
    callbacks=[
        lgb.early_stopping(stopping_rounds=200),
        lgb.log_evaluation(100)
    ]
)

y_pred = model.predict(X_val)
y_pred = np.clip(y_pred, 0, None)

rmse = np.sqrt(mean_squared_error(y_val, y_pred))
print("RMSE on validation set:", rmse)


15972
['SKU-00001' 'SKU-00002' 'SKU-00003' 'SKU-00004' 'SKU-00005']
Training until validation scores don't improve for 200 rounds
[100]	training's rmse: 4.64772	valid_1's rmse: 1.22684
[200]	training's rmse: 4.49347	valid_1's rmse: 1.31356
Early stopping, best iteration is:
[1]	training's rmse: 5.08677	valid_1's rmse: 0.0971961
RMSE: 0.09719609655165085


## 7. Khởi Tạo Dự Báo & Định Dạng File Nộp Bài
Suy luận lượng bán tương lai, áp dụng hàm chặn dưới bằng 0 và xoay ma trận sang định dạng hàng ngang khớp 100% cấu trúc yêu cầu của ban tổ chức.

In [14]:
# Select future rows
future_df = df[df["is_future"]].copy()

# Fill any remaining NaN features for inference
for f in features:
    if f != "ItemCode_cat":
        future_df[f] = future_df[f].fillna(0)

# Predict future values (predictions are already on target scale for Tweedie)
future_df["preds"] = np.clip(model.predict(future_df[features]), 0, None)

print(f"Pred stats: min={future_df['preds'].min():.4f}  mean={future_df['preds'].mean():.4f}  max={future_df['preds'].max():.2f}")
print(f"Non-zero predictions (>0.01): {(future_df['preds'] > 0.01).sum():,} / {len(future_df):,}")

pivot_df = future_df.pivot_table(
    index='ItemCode',
    columns='Date',
    values='preds',
    aggfunc='mean'
).fillna(0)

pivot_df = pivot_df.sort_index(axis=1)

val_dates = pd.date_range(TRAIN_END + pd.Timedelta(days=1), periods=28, freq="D")
eval_dates = pd.date_range(TRAIN_END + pd.Timedelta(days=29), periods=28, freq="D")

val_preds = pivot_df[val_dates].copy()
eval_preds = pivot_df[eval_dates].copy()

val_preds.columns = [f'F{i}' for i in range(1, 29)]
eval_preds.columns = [f'F{i}' for i in range(1, 29)]

val_preds = val_preds.reset_index()
eval_preds = eval_preds.reset_index()

val_preds['id'] = val_preds['ItemCode'] + '_validation'
eval_preds['id'] = eval_preds['ItemCode'] + '_evaluation'

val_preds = val_preds.drop(columns=['ItemCode'])
eval_preds = eval_preds.drop(columns=['ItemCode'])

sub = pd.concat([val_preds, eval_preds], ignore_index=True)

cols = ['id'] + [f'F{i}' for i in range(1, 29)]
sub = sub[cols]

sub.to_csv('submissions/submission.csv', index=False)
print(f"Submission shape: {sub.shape}")
print("Saved submission to submissions/submission.csv")
